# 美的集团 (000333.SZ) 数据获取与可视化

本 notebook 展示从 **Tushare Pro** 获取美的集团近一年日线行情数据，并进行可视化分析的全过程。

---
**作者**: 小果 🦞  
**数据来源**: [Tushare Pro](https://tushare.pro)  
**语言**: Python 3

## 1. 环境准备

In [ ]:
# 安装依赖（如已安装可跳过）
# !pip install tushare pandas matplotlib mplfinance

import tushare as ts
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体（Windows）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('环境就绪 ✅')

## 2. 设置 Tushare Token 并获取数据

美的集团 (Midea Group) 是中国家电行业龙头企业，A股代码为 **000333.SZ**，在深圳证券交易所上市。

In [ ]:
# 设置 Tushare Token（请替换为你自己的 token）
ts.set_token('5786743e32ed0a3472d784b70cb7a2692caf675438701623112a2d2a')
pro = ts.pro_api()

# 获取股票基本信息
df_info = pro.stock_basic(ts_code='000333.SZ', fields=['ts_code','name','area','industry','list_date'])
print(df_info.to_string(index=False))

In [ ]:
# 获取近一年日线行情数据
df = pro.daily(
    ts_code='000333.SZ',
    start_date='20250704',
    end_date='20260704',
    fields=['ts_code','trade_date','open','high','low','close','pre_close','change','pct_chg','vol','amount']
)

# 按日期升序排列
df = df.sort_values('trade_date').reset_index(drop=True)

print(f'共获取 {len(df)} 条日线记录')
print(f'日期范围: {df["trade_date"].iloc[0]} ~ {df["trade_date"].iloc[-1]}')
df.tail()

### 数据预览

In [ ]:
# 基本统计信息
print('=== 关键统计 ===')
print(f'最新收盘价: ¥{df["close"].iloc[-1]:.2f}')
print(f'区间最高价: ¥{df["high"].max():.2f}')
print(f'区间最低价: ¥{df["low"].min():.2f}')
print(f'区间涨幅: {(df["close"].iloc[-1] / df["close"].iloc[0] - 1) * 100:.2f}%')
print(f'日均成交量: {df["vol"].mean() / 10000:.0f} 万手')
print(f'日均成交额: ¥{df["amount"].mean() / 10000:.0f} 万')

## 3. 保存数据到本地

In [ ]:
# 保存为 CSV
df.to_csv('midea_daily.csv', index=False, encoding='utf-8-sig')
print('CSV 已保存: midea_daily.csv')

# 保存为 JSON
df.to_json('midea_daily.json', orient='records', force_ascii=False)
print('JSON 已保存: midea_daily.json')

## 4. 价格走势图

绘制收盘价和成交量的双轴走势图。

In [ ]:
# 转换日期格式
df['trade_date_dt'] = pd.to_datetime(df['trade_date'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle('美的集团 (000333.SZ) 近一年行情走势', fontsize=16, fontweight='bold', y=0.98)

# 子图1: 收盘价 + 5日/20日均线
df['ma5'] = df['close'].rolling(5).mean()
df['ma20'] = df['close'].rolling(20).mean()

ax1.plot(df['trade_date_dt'], df['close'], label='收盘价', color='#333', linewidth=1.5)
ax1.plot(df['trade_date_dt'], df['ma5'], label='MA5', color='#e74c3c', linewidth=1, linestyle='--')
ax1.plot(df['trade_date_dt'], df['ma20'], label='MA20', color='#3498db', linewidth=1, linestyle='--')
ax1.set_ylabel('价格 (¥)', fontsize=12)
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)

# 子图2: 成交量（红涨绿跌）
colors = ['#e74c3c' if c >= o else '#27ae60' for c, o in zip(df['close'], df['open'])]
ax2.bar(df['trade_date_dt'], df['vol'] / 10000, color=colors, width=1)
ax2.set_ylabel('成交量 (万手)', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 使用 mplfinance 绘制专业 K 线图

[mplfinance](https://github.com/matplotlib/mplfinance) 是一个专门用来画 K 线图的库。

In [ ]:
import mplfinance as mpf

# 准备 OHLC 数据
df_ohlc = df.set_index('trade_date_dt')
df_ohlc = df_ohlc[['open', 'high', 'low', 'close', 'vol']]
df_ohlc.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

# 自定义 K 线颜色（A股规范：红涨绿跌）
mc = mpf.make_marketcolors(
    up='#e74c3c',     # 涨 = 红色
    down='#27ae60',   # 跌 = 绿色
    edge='inherit',
    volume='inherit'
)
s = mpf.make_mpf_style(marketcolors=mc, gridstyle=':', y_on_right=False)

# 绘图
fig, axes = mpf.plot(
    df_ohlc,
    type='candle',
    volume=True,
    style=s,
    figsize=(14, 8),
    title='美的集团 (000333.SZ) K线图',
    ylabel='价格 (¥)',
    ylabel_lower='成交量 (手)',
    mav=(5, 20),          # 叠加 5日/20日均线
    returnfig=True
)

plt.show()

## 6. 涨跌幅分布分析

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 涨跌幅直方图
axes[0].hist(df['pct_chg'], bins=30, color='#3498db', edgecolor='white', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('涨跌幅 (%)')
axes[0].set_ylabel('天数')
axes[0].set_title('日涨跌幅分布')
axes[0].grid(True, alpha=0.3)

# 涨跌天数统计
up_days = len(df[df['pct_chg'] > 0])
down_days = len(df[df['pct_chg'] < 0])
flat_days = len(df[df['pct_chg'] == 0])

axes[1].pie([up_days, down_days, flat_days],
            labels=[f'上涨 {up_days}天', f'下跌 {down_days}天', f'平盘 {flat_days}天'],
            colors=['#e74c3c', '#27ae60', '#95a5a6'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('涨跌天数比例')

plt.tight_layout()
plt.show()

## 7. 波动率分析

In [ ]:
# 日收益率
df['daily_return'] = df['close'].pct_change()

# 年化波动率
daily_vol = df['daily_return'].std()
annual_vol = daily_vol * np.sqrt(252)

# 日均振幅
avg_amplitude = (df['high'] - df['low']).mean()
avg_amplitude_pct = ((df['high'] - df['low']) / df['close'] * 100).mean()

print('=== 风险指标 ===')
print(f'日收益率标准差: {daily_vol:.4f} ({daily_vol*100:.2f}%)')
print(f'年化波动率: {annual_vol:.4f} ({annual_vol*100:.2f}%)')
print(f'日均振幅: ¥{avg_amplitude:.2f} ({avg_amplitude_pct:.2f}%)')
print(f'最大单日涨幅: {df["pct_chg"].max():.2f}%')
print(f'最大单日跌幅: {df["pct_chg"].min():.2f}%')

## 8. 总结

通过本 notebook，我们完成了以下工作：

1. ✅ 使用 Tushare Pro API 获取美的集团 A 股近一年日线数据
2. ✅ 将数据保存为 CSV 和 JSON 格式
3. ✅ 绘制收盘价走势图（含 MA5/MA20 均线）
4. ✅ 绘制专业 K 线图 + 成交量柱状图
5. ✅ 分析涨跌幅分布和波动率

美的集团作为家电龙头，近一年股价走势体现出低波动、稳增长的特征，适合作为价值投资策略的候选标的。